# Swin, an adaptive attacker, and the cost of the defence

Three questions the ViT results do not answer on their own. Does the ranking transfer to an architecture whose attention is windowed rather than global? What happens when the attacker knows PSBD exists and trains against it? And what does the defence actually cost, in forward passes, against a plain prediction? All three read cached results rather than training or sweeping anything new.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in the library resolves the
# same way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import logging

logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import scripts.paper._style  # noqa: F401  the figure style every table in the paper uses

import glob
import json

from defences.decision import PUBLISHED_PLACEMENT, RECOMMENDED_PLACEMENT
from scripts.paper._common import bootstrap_ci, mean_or_none

## Transfer to Swin-S

Swin-S restricts attention to shifted windows and merges patches between 4 stages, so tokens mix locally within a stage and reach every other token only through merging. If routing through attention is really what makes token masking work, the ranking should survive that restriction. This reads every `swin_*` checkpoint's `psbd_metrics.json` directly, the same files `scripts/paper/tab_swin.py` reads, filtered to the models whose attack success clears the bar.

In [2]:
SWIN_PLACEMENTS = {
    "token mask, attention input (PSBD-TM)": RECOMMENDED_PLACEMENT,
    "dropout, attention input": "before_attention_norm",
    "dropout, after both residual adds (PSBD-RD)": PUBLISHED_PLACEMENT,
}
ASR_BAR = 0.85

swin_rows = {label: [] for label in SWIN_PLACEMENTS}
for path in sorted(glob.glob("results/swin_*/psbd_metrics.json")):
    folder = path.split("/")[1]
    if "benign" in folder or any(tag in folder for tag in ("_sam_", "_seed_", "_evade_")):
        continue
    args_path = f"checkpoints/{folder}/args.json"
    if not os.path.exists(args_path):
        continue
    with open(args_path) as handle:
        asr = json.load(handle).get("asr")
    if asr is None or asr < ASR_BAR:
        continue
    with open(path) as handle:
        report = json.load(handle)
    for label, placement_id in SWIN_PLACEMENTS.items():
        block = report.get("placements", {}).get(placement_id)
        rate = block.get("adaptive_rate") if block else None
        if rate is None:
            continue
        rate_row = next((r for r in block["rates"] if r["rate"] == rate), None)
        if rate_row is None:
            continue
        swin_rows[label].append(rate_row["detection_psu_ratio"]["q0.25"]["auroc"])

swin_table = pd.DataFrame(
    [{"placement": label, "n": len(values), "auroc": mean_or_none(values)} for label, values in swin_rows.items()]
).set_index("placement")
swin_table.round(3)

,n,auroc
placement,,
"token mask, attention input (PSBD-TM)",53,0.968
"dropout, attention input",59,0.862
"dropout, after both residual adds (PSBD-RD)",60,0.871


The ranking transfers. Token masking at the attention input beats both dropout placements on Swin by roughly the same margin it does on ViT, and moving dropout to the attention input alone, without changing the operator, gains far less than switching the operator does. Windowing the attention does not change which axis, position or operator, carries the result.

## An attacker who trains against the probe

Every result so far assumes an attacker who does not know PSBD exists. `results/adaptive_attacker_analysis.json` carries 15 ViT-B/16 models trained on CIFAR-100 with a hinge loss added to the backdoor objective, pushing a poisoned sample's PSU toward a clean sample's PSU under PSBD-TM specifically, jointly with the backdoor itself.

In [3]:
with open("results/adaptive_attacker_analysis.json") as handle:
    attacker = json.load(handle)

attacker_frame = pd.DataFrame(attacker)
attacker_frame["cost"] = attacker_frame["evade_ca"] - attacker_frame["base_ca"]

summary = pd.DataFrame(
    {
        "probed placement (token mask, attention input), before training against it":
            [attacker_frame["token_mask_at_ban_base"].mean()],
        "same placement, after training against it": [attacker_frame["token_mask_at_ban_evade"].mean()],
        "dropout at the attention input, never trained against": [attacker_frame["dropout_at_ban_evade"].mean()],
        "gain scale at the MLP norm output, never trained against": [attacker_frame["gain_scale_at_mno_evade"].mean()],
        "mean clean-accuracy cost of the evasion": [attacker_frame["cost"].mean()],
    }
).T
summary.columns = ["mean AUROC / cost"]
summary.round(3)

,mean AUROC / cost
"probed placement (token mask, attention input), before training against it",0.854
"same placement, after training against it",0.265
"dropout at the attention input, never trained against",0.627
"gain scale at the MLP norm output, never trained against",0.797
mean clean-accuracy cost of the evasion,-0.030


Against the exact placement it trains against, the attacker wins outright, AUROC falls from close to 1 to near chance. The two probes it never saw during training keep most of their separation, because the attacker's gradient only touches the features the one probe it trains against reads. This is the argument for a defence built from more than one probe.

## The union of probes

`results/multi_probe_analysis.json` scores the same evasive models with combinations of probes, using the min-rank union: score an input by its worst rank across probes, so it escapes detection only when every probe in the union considers it clean.

In [4]:
with open("results/multi_probe_analysis.json") as handle:
    union = json.load(handle)

union_frame = pd.DataFrame(union)
union_summary = pd.DataFrame(
    {
        "probed placement alone": union_frame["single_tm@ban_auroc"],
        "2 probes (token mask + gain scale)": union_frame["combo_tm@ban+gs@mno_auroc"],
        "3 probes (token mask + dropout + gain scale)": union_frame["combo_tm@ban+do@ban+gs@mno_auroc"],
    }
).mean()
union_summary.round(3)

probed placement alone                          0.265
2 probes (token mask + gain scale)              0.759
3 probes (token mask + dropout + gain scale)    0.753
dtype: float64

The union restores most of what the single probed placement lost. Nothing here trains against the union itself, so this is a defence against an attacker who targets one known probe, not against one who targets the union, which the paper states as an open question.

## Forward passes and their cost

PSBD's price is one extra forward pass per perturbed prediction. `experiments/psbd_cost/measure.py` timed a plain prediction against PSBD-TM at several pass counts, batch 128, on the login-node A100, and `results/_experiments/psbd_cost/cost.json` carries the result.

In [5]:
with open("results/_experiments/psbd_cost/cost.json") as handle:
    cost = json.load(handle)

vit_cost = cost["architectures"]["vit"]
plain_seconds = vit_cost["plain"]["seconds_per_input"]
placement_block = vit_cost["placements"]["token_mask_before_attention_norm"]["by_k"]

cost_rows = [
    {
        "k": int(k),
        "seconds_per_input": row["seconds_per_input"],
        "times_a_plain_prediction": row["seconds_per_input"] / plain_seconds,
    }
    for k, row in placement_block.items()
]
pd.DataFrame(cost_rows).set_index("k").round(4)

,seconds_per_input,times_a_plain_prediction
k,,
1,0.0022,1.0074
3,0.0067,3.0432
5,0.0112,5.0999
10,0.0226,10.3069
20,0.0460,20.9379


At k = 3, the paper's own choice, PSBD costs roughly 3 times a plain prediction. Pushing k to 20 buys very little further separation, `paper/sections/08-robustness.tex` reports 0.003 AUROC for 7 times the cost of k = 3, so k = 3 is close to where the trade stops paying for itself.